# NB-R07 — SHAP Regime-Specific Feature Importance

**Pipeline stage:** 7 of 13

**Purpose.** Quantify which engineered features drive the XGBoost base learner's predictions, separately within the High-VIX and Low-VIX subsets, using SHAP (SHapley Additive exPlanations) values rather than a qualitative description.

**Inputs:** `data/processed/test_with_regimes.csv`, `models/xgb_model.joblib`.

**Outputs:** `results/shap_importance_High_VIX.csv`, `shap_importance_Low_VIX.csv`, `shap_importance_Overall.csv`, `shap_regime_shift.csv`, `plots/R07_shap_beeswarm_high_vix.png`, `plots/R07_shap_regime_comparison.png`.

**Result:** `bb_upper` (Bollinger upper band) is the dominant feature in both regimes. Beyond that, the regime shift is feature-specific rather than a uniform amplification of volatility-sensitive indicators: `macd_signal`, `ema20`, and `macd` become relatively more important in the High-VIX subset, while `atr14` and `bb_width` -- the two features most directly built to measure realized dispersion -- become markedly *less* important there than in the Low-VIX subset. All High-VIX-subset SHAP values are computed on only 8 observations and should be read with that caveat.


In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

PROJ    = Path('..').resolve()  # repo root, assuming this notebook is run from notebooks/
PROC    = PROJ / 'data' / 'processed'
RESULTS = PROJ / 'results'
PLOTS   = PROJ / 'plots'
MODELS  = PROJ / 'models'

with open(PROC / 'feature_cols.json') as f:
    FEATURE_COLS = json.load(f)

test  = pd.read_csv(PROC / 'test_predictions.csv', parse_dates=['date'])
train = pd.read_csv(PROC / 'train.csv', parse_dates=['date'])

X_train = train[FEATURE_COLS].values
X_test  = test[FEATURE_COLS].values
regime  = test['regime_fixed'].values

xgb_model = joblib.load(MODELS / 'xgb_model.joblib')
print('XGBoost model loaded.')
print(f'Test shape: {X_test.shape}')

## 1. SHAP Values via TreeExplainer (XGBoost)

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# shap_values shape: (n_test, n_features)
shap_df = pd.DataFrame(shap_values, columns=FEATURE_COLS)
shap_df['regime'] = regime

print(f'SHAP values shape: {shap_values.shape}')
print('SHAP computation complete.')

## 2. Regime-Specific Feature Importance

In [ ]:
importance_by_regime = {}
for reg in ['High-VIX', 'Low-VIX', 'Overall']:
    if reg == 'Overall':
        sub = shap_df[FEATURE_COLS]
    else:
        sub = shap_df[shap_df['regime'] == reg][FEATURE_COLS]

    mean_abs = sub.abs().mean().sort_values(ascending=False)
    importance_by_regime[reg] = mean_abs
    print(f'\n--- {reg} (n={len(sub)}) --- Mean |SHAP| ranking:')
    for feat, val in mean_abs.head(10).items():
        print(f'  {feat:<25} {val:.5f}')

In [ ]:
# Save importance tables
for reg, imp in importance_by_regime.items():
    imp.reset_index().rename(columns={'index': 'feature', 0: 'mean_abs_shap'}).to_csv(
        RESULTS / f'shap_importance_{reg.replace("-","_")}.csv', index=False)
print('SHAP importance tables saved.')

## 3. Comparison Plot: High-VIX vs Low-VIX Feature Rankings

In [ ]:
top_features = importance_by_regime['Overall'].head(12).index.tolist()

high_vals  = importance_by_regime['High-VIX'][top_features]
low_vals   = importance_by_regime['Low-VIX'][top_features]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(top_features))
ax.barh(x + 0.2, high_vals.values, height=0.35, label='High-VIX', color='tomato', alpha=0.85)
ax.barh(x - 0.2, low_vals.values,  height=0.35, label='Low-VIX',  color='steelblue', alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(top_features)
ax.set_xlabel('Mean |SHAP| Value')
ax.set_title('Regime-Specific SHAP Feature Importance (XGBoost, 21d Horizon)')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS / 'R07_shap_regime_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP comparison plot saved.')

## 4. SHAP Beeswarm Plot (High-VIX only)

In [ ]:
high_mask = regime == 'High-VIX'
shap_high = shap_values[high_mask]
X_test_high = X_test[high_mask]

shap.summary_plot(
    shap_high, X_test_high,
    feature_names=FEATURE_COLS,
    max_display=12,
    show=False,
    plot_type='dot'
)
plt.title('SHAP Beeswarm: High-VIX Regime')
plt.tight_layout()
plt.savefig(PLOTS / 'R07_shap_beeswarm_high_vix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Beeswarm plot saved.')

## 5. Feature Change Analysis: Which Features Shift Most Between Regimes?

In [ ]:
shift = (importance_by_regime['High-VIX'] - importance_by_regime['Low-VIX']).sort_values(ascending=False)
print('Features with largest SHAP increase in High-VIX vs Low-VIX:')
print(shift.head(8).to_string())
print('\nFeatures with largest SHAP decrease in High-VIX vs Low-VIX:')
print(shift.tail(8).to_string())

shift.to_csv(RESULTS / 'shap_regime_shift.csv', header=['shap_diff_high_minus_low'])
print('Regime shift saved.')

---
## Summary

**Pipeline stage:** 7 of 13 (see `notebooks/README.md` for the full pipeline map).

**Artifacts produced by this notebook:**

- `results/shap_importance_High_VIX.csv`
- `results/shap_importance_Low_VIX.csv`
- `results/shap_importance_Overall.csv`
- `results/shap_regime_shift.csv`
- `plots/R07_shap_beeswarm_high_vix.png`
- `plots/R07_shap_regime_comparison.png`

**Next notebook:** `NB-R08_ablation_studies.ipynb`
